# Station Portfolio Analysis

**Purpose:** Used for `Section 2.1.2.1 in Report`. 

Also generate additional characteristics (i.e., more columns) and produce a complete survey dataset for further analysis. A spreadsheet is also provided, containing the full dataset along with pivot table analyses that directly support the report. See `4_2_Rider_Portfolio_by_Station_Archive.xlsx` for reference.

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [ ]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df = pd.read_csv(interim_dir + "Interim_2023_OnBoardSurvey_Dataset_3.csv")

### Define Other Characteristics

by Age

In [3]:
df['YEAR_BORN'] = df['YEAR_BORN'].fillna(9999)
df['AGE'] = 2023-df['YEAR_BORN']

In [4]:
df['AGE_GROUP'] = np.where((df['AGE']>=0) & (df['AGE']<18), "under 18", 
                  np.where((df['AGE']>=18) & (df['AGE']<23), "18-23", 
                  np.where((df['AGE']>=23) & (df['AGE']<28), "23-28", 
                  np.where((df['AGE']>=28) & (df['AGE']<35), "28-35", 
                  np.where((df['AGE']>=35) & (df['AGE']<45), "35-45",
                  np.where((df['AGE']>=45) & (df['AGE']<65), "45-65",
                  np.where(df['AGE']>=65, "65 above",
                  None)))))))

by Income Group

In [5]:
df['INCOME_GROUP7'] = np.where(df['INCOME']=='$14,999 or less', "under $15k", 
                      np.where((df['INCOME']=='$15,000 - $19,999') | (df['INCOME']=='$20,000 - $24,999'), "$15k-$25k", 
                      np.where((df['INCOME']=='$25,000 - $29,999') | (df['INCOME']=='$30,000 - $34,999') | (df['INCOME']=='$35,000 - $39,999')
                             | (df['INCOME']=='$40,000 - $44,999') | (df['INCOME']=='$45,000 - $49,999'), "$25k-$50k", 
                     
                      np.where((df['INCOME']=='$50,000 - $59,999') | (df['INCOME']=='$60,000 - $74,999'), "$50k-$75k", 
                      np.where(df['INCOME']=='$75,000 - $99,999', "$75k-$100k",
                      np.where(df['INCOME']=='$100,000 - $149,999', "$100k-$150k",
                      np.where(df['INCOME']=='$150,000 or above', "$150k above",
                      "Refused/No Answer")))))))

In [6]:
df['INCOME_GROUP3'] = np.where((df['INCOME_GROUP7']=='under $15k') | (df['INCOME_GROUP7']=='$15k-$25k') | (df['INCOME_GROUP7']=='$25k-$50k') , "1LowInc", 
                      np.where((df['INCOME_GROUP7']=='$50k-$75k') | (df['INCOME_GROUP7']=='$75k-$100k'), "2MedInc", 
                      np.where((df['INCOME_GROUP7']=='$100k-$150k') | (df['INCOME_GROUP7']=='$150k above'), "3HighInc", 
                      "Refused/No Answer")))

by Time

In [7]:
# check NAN/blank value
nan_count = df['TIME_ON'].isna().sum()
blank_count = (df['TIME_ON'] == '').sum()

print(f"NaN values: {nan_count}")
print(f"Blank strings: {blank_count}")

NaN values: 0
Blank strings: 0


In [8]:
df['TIME_ON_GROUP'] = np.where((df['TIME_ON']=='6:00 am - 6:59 am') | (df['TIME_ON']=='7:00 am - 7:59 am') | (df['TIME_ON']=='8:00 am - 8:59 am')
                             | (df['TIME_ON']=='3:00 pm - 3:59 pm') | (df['TIME_ON']=='4:00 pm - 4:59 pm') | (df['TIME_ON']=='5:00 pm - 5:59 pm')
                             | (df['TIME_ON']=='6:00 pm - 6:59 pm'), "PK", 

                      np.where((df['TIME_ON']=='9:00 am - 9:59 am') | (df['TIME_ON']=='10:00 am - 10:59 am') | (df['TIME_ON']=='11:00 am - 11:59 am')
                             | (df['TIME_ON']=='12:00 pm - 12:59 pm') | (df['TIME_ON']=='1:00 pm - 1:59 pm') | (df['TIME_ON']=='2:00 pm - 2:59 pm'), "OP", 
                      
                      np.where((df['TIME_ON']=='Before 6:00 am') | (df['TIME_ON']=='7:00 pm - 7:59 pm') | (df['TIME_ON']=='8:00 pm - 8:59 pm')
                             | (df['TIME_ON']=='After 9:00 pm'), "EVE/NT",
                      
                      None)))

by Insufficient Auto Ownership Household

In [9]:
df['INSUFFI'] = df['COUNT_VH_HH[Code]'] - df['EMPLOYED_IN_HH[Code]']

In [10]:
df['INSUFFI_GROUP'] = np.where(df['INSUFFI']<0 , "1 less 0", 
                      np.where(df['INSUFFI']==0, "2 equal to 0", 
                      np.where(df['INSUFFI']>0, "3 above 0", 
                      None)))

### Generate Full Dataset

In [11]:
df.to_csv(output_dir + 'Final_2023_OnBoardSurvey_Dataset.csv',index=False)

### Run

In [12]:
project_trips = df[df['TRIPS_ON_PROJECT']=='Yes']

In [13]:
station_names = [
    'UTC Station',
    'Executive Drive Station',
    'UCSD Health La Jolla Station',
    'UCSD Central Campus Station',
    'VA Medical Center Station',
    'Nobel Drive Station',
    'Balboa Avenue Station',
    'Clairemont Drive Station',
    'Tecolote Road Station'
]

Define the route direction, project station, and whether the focus is on boardings or alightings. Once defined, you can extract the relevant characteristics for that station.

For example, the code below shows student rider information for northbound boardings at Nobel Drive Station.

In [14]:
# Define route direction: 0 is NorthBound and 1 is SouthBound
dir = 0
# Define project station
stop = 'Nobel Drive Station'
# Define boardings or alightings: STOP_Prod or STOP_Attr
stop_type = 'STOP_Prod'
# Define characteristics, e.g.:'EMPLOYMENT_STATUS', 'TRIP_PURPOSE', 'ACCESS_MODE_PA' ,'INCOME_GROUP3', 'AGE_GROUP'
category = 'STUDENT_STATUS'

project_trips[(project_trips['Direction_PA'] == dir) & (project_trips[stop_type] == stop)].groupby(category).agg({'REWEIGHTED_LINKED': 'sum'})\
        .assign(Percentage=lambda x: (x['REWEIGHTED_LINKED'] / x['REWEIGHTED_LINKED'].sum()) * 100).round(1)



,REWEIGHTED_LINKED,Percentage
STUDENT_STATUS,,
Not a student,1095.4,41.2
Yes - Full-time College / University,1174.2,44.2
Yes - Part-time College / University,386.2,14.5


In [15]:
# In a Loop, to print information for all project stations in a specific boarding direction.
dir = 1
stop_type = 'STOP_Prod'
category = 'INCOME_GROUP3'

for stop in station_names:
    print(stop)
    print(project_trips[(project_trips['Direction_PA'] == dir) & (project_trips[stop_type] == stop)].groupby(category).agg({'REWEIGHTED_LINKED': 'sum'})\
            .assign(Percentage=lambda x: (x['REWEIGHTED_LINKED'] / x['REWEIGHTED_LINKED'].sum()) * 100).round(1))
    print("---")

UTC Station
                   REWEIGHTED_LINKED  Percentage
INCOME_GROUP3                                   
1LowInc                       1044.8        41.4
2MedInc                        724.6        28.7
3HighInc                       261.3        10.3
Refused/No Answer              495.2        19.6
---
Executive Drive Station
                   REWEIGHTED_LINKED  Percentage
INCOME_GROUP3                                   
1LowInc                        706.6        49.5
2MedInc                        489.1        34.2
3HighInc                        85.6         6.0
Refused/No Answer              147.3        10.3
---
UCSD Health La Jolla Station
               REWEIGHTED_LINKED  Percentage
INCOME_GROUP3                               
1LowInc                    132.8        20.3
2MedInc                    510.0        77.8
3HighInc                    12.8         2.0
---
UCSD Central Campus Station
                   REWEIGHTED_LINKED  Percentage
INCOME_GROUP3                    